In [2]:
import tensorflow as tf

MODEL_PATHS = {
    "Model A": "/kaggle/input/models/hemasakshis/comaprisons/tensorflow2/default/1/finetuned_35_layers.keras",

    "Model B": "/kaggle/input/models/hemasakshis/comaprisons/tensorflow2/default/1/finetuned_60_layers.keras",

    "Model C": "/kaggle/input/models/hemasakshis/comaprisons/tensorflow2/default/1/final_dr_model.keras",

    "Model D": "/kaggle/input/models/hemasakshis/comaprisons/tensorflow2/default/1/final_experiment_D.keras"
}

models = {}

for name, path in MODEL_PATHS.items():
    models[name] = tf.keras.models.load_model(path)
    print(f"{name} loaded successfully!")

Model A loaded successfully!
Model B loaded successfully!
Model C loaded successfully!
Model D loaded successfully!


In [3]:
for name, model in models.items():
    print("\n", name)
    print("Input shape:", model.input_shape)
    print("Output shape:", model.output_shape)
    print("Total parameters:", model.count_params())


 Model A
Input shape: (None, 224, 224, 3)
Output shape: (None, 5)
Total parameters: 23597957

 Model B
Input shape: (None, 224, 224, 3)
Output shape: (None, 5)
Total parameters: 23597957

 Model C
Input shape: (None, 224, 224, 3)
Output shape: (None, 5)
Total parameters: 23597957

 Model D
Input shape: (None, 224, 224, 3)
Output shape: (None, 5)
Total parameters: 23597957


In [4]:
import os

for item in os.listdir("/kaggle/input"):
    print(item)

models
competitions


In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split

train_df = pd.read_csv(
    "/kaggle/input/competitions/aptos2019-blindness-detection/train.csv"
)

# 70% training, 30% temporary
train_data, temp_data = train_test_split(
    train_df,
    test_size=0.30,
    stratify=train_df["diagnosis"],
    random_state=42
)

# Split remaining 30% equally into validation and test
val_data, test_data = train_test_split(
    temp_data,
    test_size=0.50,
    stratify=temp_data["diagnosis"],
    random_state=42
)

print("Training images:", len(train_data))
print("Validation images:", len(val_data))
print("Test images:", len(test_data))

Training images: 2563
Validation images: 549
Test images: 550


In [6]:
import tensorflow as tf
import os

IMAGE_DIR = "/kaggle/input/competitions/aptos2019-blindness-detection/train_images"

IMG_SIZE = 224
BATCH_SIZE = 32

test_data = test_data.copy()

test_data["image_path"] = test_data["id_code"].apply(
    lambda x: os.path.join(IMAGE_DIR, x + ".png")
)

def preprocess_image(image_path, label):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_png(image, channels=3)
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = tf.keras.applications.resnet50.preprocess_input(image)
    return image, label

test_dataset = tf.data.Dataset.from_tensor_slices((
    test_data["image_path"].values,
    test_data["diagnosis"].values
))

test_dataset = (
    test_dataset
    .map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

print("Test dataset prepared successfully!")
print("Number of test images:", len(test_data))

Test dataset prepared successfully!
Number of test images: 550


In [7]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    cohen_kappa_score
)

y_true = test_data["diagnosis"].values

results = []

for name, model in models.items():

    print(f"Evaluating {name}...")

    y_prob = model.predict(test_dataset, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)

    results.append({
        "Model": name,
        "Test Accuracy (%)": round(accuracy_score(y_true, y_pred) * 100, 2),
        "Precision (Macro)": round(precision_score(y_true, y_pred, average="macro", zero_division=0), 4),
        "Recall (Macro)": round(recall_score(y_true, y_pred, average="macro", zero_division=0), 4),
        "Macro F1": round(f1_score(y_true, y_pred, average="macro", zero_division=0), 4),
        "Weighted F1": round(f1_score(y_true, y_pred, average="weighted", zero_division=0), 4),
        "QWK": round(cohen_kappa_score(y_true, y_pred, weights="quadratic"), 4)
    })

comparison_df = pd.DataFrame(results)

print("\nFINAL FOUR-MODEL COMPARISON")
display(comparison_df)

Evaluating Model A...
Evaluating Model B...
Evaluating Model C...
Evaluating Model D...

FINAL FOUR-MODEL COMPARISON


,Model,Test Accuracy (%),Precision (Macro),Recall (Macro),Macro F1,Weighted F1,QWK
0,Model A,77.82,0.5838,0.5847,0.5835,0.7781,0.8604
1,Model B,77.64,0.6040,0.6080,0.5940,0.7772,0.8663
2,Model C,75.82,0.5903,0.6059,0.5897,0.7641,0.8606
3,Model D,76.00,0.5801,0.5838,0.5771,0.7643,0.8403


In [8]:

import tensorflow as tf

MODEL_B_PATH = "/kaggle/input/models/hemasakshis/comaprisons/tensorflow2/default/1/finetuned_60_layers.keras"

model_b = tf.keras.models.load_model(MODEL_B_PATH)

print("Model B loaded successfully!")
print("Input shape:", model_b.input_shape)
print("Output shape:", model_b.output_shape)


Model B loaded successfully!
Input shape: (None, 224, 224, 3)
Output shape: (None, 5)
